In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

In [ ]:
!pip install pandas keras tensorflow matplotlib scikit-learn opencv-python

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Conv2D, Flatten, Dense, MaxPooling2D,Dropout,BatchNormalization
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import cv2
import os
from sklearn.utils import shuffle
import numpy as np
print("done")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
labels = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
x_train = []
y_train = []

# TODO: Replace with the actual path to your training dataset in Google Drive
training_data_path = "/content/drive/MyDrive/project brain tumor /dataset/Training"
testing_data_path = "/content/drive/MyDrive/project brain tumor /dataset/Testing"

for i in labels:
    folder_path = os.path.join(training_data_path, i)
    for j in os.listdir(folder_path):
        image = cv2.imread(os.path.join(folder_path, j))
        image = cv2.resize(image, (150, 150))
        image = image / 255.0  # <-- Normalize the image here
        x_train.append(image)
        y_train.append(i)

print("done 1")

for i in labels:
    folder_path = os.path.join(testing_data_path, i)
    for j in os.listdir(folder_path):
        image = cv2.imread(os.path.join(folder_path, j))
        image = cv2.resize(image, (150, 150))
        image = image / 255.0  # <-- Normalize the image here
        x_train.append(image)
        y_train.append(i)

print("done")

In [ ]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_train, y_train = shuffle(x_train,y_train,random_state = 101)
x_train.shape

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(x_train,y_train,test_size = 0.1,random_state = 39)
X_train.shape

In [9]:
Y_train_new = []
Y_test_new = []

for i in Y_train:
    Y_train_new.append(labels.index(i))

Y_train = Y_train_new

for i in Y_test:
    Y_test_new.append(labels.index(i))

Y_test = Y_test_new



In [ ]:
len(Y_train_new)

In [11]:
import numpy as np

Y_train = np.array(Y_train)
Y_test = np.array(Y_test)


In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

In [13]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
model.summary()

In [ ]:
history = model.fit(X_train, Y_train, epochs=50, validation_data=(X_test, Y_test))

In [ ]:
from sklearn.metrics import precision_score, recall_score

In [ ]:
model.evaluate(X_test, Y_test)

# Predict the output for the test data
Y_pred = model.predict(X_test)

# Convert the predicted probabilities to class labels
Y_pred_labels = np.argmax(Y_pred, axis=1)

# Define precision, recall, and accuracy metrics
precision = tf.keras.metrics.Precision()
recall = tf.keras.metrics.Recall()
accuracy = tf.keras.metrics.Accuracy()

# Update the state of the metrics with the true and predicted labels
precision.update_state(Y_test, Y_pred_labels)
recall.update_state(Y_test, Y_pred_labels)
accuracy.update_state(Y_test, Y_pred_labels)

# Print the results
print("Precision:", precision.result().numpy())
print("Recall:", recall.result().numpy())
print("Accuracy:", accuracy.result().numpy())


In [ ]:
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(Y_test, Y_pred_labels)


In [ ]:
conf_matrix

In [ ]:
conf_matrix

In [ ]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, Y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Predict classes for test set
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

# Initialize counters for correct and wrong predictions for each class
correct_predictions = [0, 0, 0, 0]
wrong_predictions = [0, 0, 0, 0]

# Count correct and wrong predictions for each class
for i in range(len(Y_test)):
    true_label = Y_test[i]
    predicted_label = predicted_classes[i]
    if true_label == predicted_label:
        correct_predictions[true_label] += 1
    else:
        wrong_predictions[true_label] += 1

# Print results
for i, label in enumerate(labels):
    print("Class:", label)
    print("Correct Predictions:", correct_predictions[i])
    print("Wrong Predictions:", wrong_predictions[i])
    print("")

# Total Correct and Wrong Predictions
total_correct = sum(correct_predictions)
total_wrong = sum(wrong_predictions)
print("Total Correct Predictions:", total_correct)
print("Total Wrong Predictions:", total_wrong)


In [ ]:
model.save("/content/drive/MyDrive/model.h5")

In [ ]:
from keras.models import load_model

In [ ]:
l_model=load_model("/content/drive/MyDrive/model.h5")

In [ ]:
l_model.summary()

In [ ]:
test_folder = "/content/drive/MyDrive/project brain tumor /dataset/Testing"

# Initialize counts for correct and wrong predictions for each class
correct_predictions = [0, 0, 0, 0]
wrong_predictions = [0, 0, 0, 0]

# Iterate through each class folder
for i, label in enumerate(labels):
    class_folder = os.path.join(test_folder, label)

    # Iterate through each image in the class folder
    for image_name in os.listdir(class_folder):
        image_path = os.path.join(class_folder, image_name)

        # Read and preprocess the image
        image = cv2.imread(image_path)
        image = cv2.resize(image, (150, 150))
        image = image.reshape(1, 150, 150, 3)  # Reshape for model prediction

        # Predict the class of the image
        prediction = model.predict(image)
        predicted_class = np.argmax(prediction)

        # Check if prediction is correct or wrong
        if predicted_class == i:  # Predicted correctly
            correct_predictions[i] += 1
        else:  # Predicted wrongly
            wrong_predictions[i] += 1

# Print correct and wrong predictions for each class
for i, label in enumerate(labels):
    print("Class:", label)
    print("Correct Predictions:", correct_predictions[i])
    print("Wrong Predictions:", wrong_predictions[i])
    print("")

In [ ]:
import cv2
import numpy as np
from keras.models import load_model


saved_model = load_model('/content/drive/MyDrive/model.h5')


labels = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]


image_path = r"/content/drive/MyDrive/brain tumor trial1/dataset/Testing/glioma/image(1).jpg"  # Ensure this path is correct


img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Image not found at {image_path}")


img = cv2.resize(img, (150, 150))
img = np.expand_dims(img, axis=0)


predictions = saved_model.predict(img)
predicted_class = np.argmax(predictions, axis=1)[0]

print("Predicted Class:", labels[predicted_class])
print(predictions)

In [ ]:
import gradio as gr
import numpy as np
import cv2
from tensorflow.keras.models import load_model
!pip install gradio

# Load your trained model
model = load_model("/content/drive/MyDrive/model.h5")

# Define the labels (in the same order as your training)
labels = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

# Define a preprocessing + prediction function
def predict_brain_tumor(image):
    try:
        # Convert PIL image to NumPy array
        img = np.array(image)

        # Resize and preprocess
        img = cv2.resize(img, (150, 150))
        img = np.expand_dims(img, axis=0)

        # Predict
        predictions = model.predict(img)
        class_index = np.argmax(predictions)
        confidence = float(np.max(predictions)) * 100

        return f"Prediction: {labels[class_index]} (Confidence: {confidence:.2f}%)"

    except Exception as e:
        return f"Error during prediction: {str(e)}"

# Create Gradio interface
interface = gr.Interface(
    fn=predict_brain_tumor,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Brain Tumor Detection (CNN)",
    description="Upload an MRI brain scan to detect tumor type using a trained CNN model."
)

# Launch
interface.launch(debug=True)